<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/08_NeuroFHIR_QC_Human_Review_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/08_NeuroFHIR_QC_Human_Review_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 08
## Human Review Workflow: accept, correction-required, reject, status transitions, provenance, validation, write-back, and read-back

Run this notebook **from top to bottom in a fresh Colab runtime**.

This notebook closes the human-in-the-loop portion of the competition workflow using the three synthetic demonstration cases and the FHIR evidence generated by Notebook 07.

### Scripted review path

| Case | Review path | Final state |
|---|---|---|
| Stable | requested → accepted | Observation and DiagnosticReport become `final`; Task becomes `completed` |
| Progression | requested → accepted | Observation and DiagnosticReport become `final`; Task becomes `completed` |
| Low confidence | requested → correction-required → rejected | correction-required is preserved as an on-hold intermediate state; the original AI result is finally marked `entered-in-error`; Task becomes `rejected` |

### Safety and interpretation boundary

- The reviewer is an explicitly **synthetic research reviewer**, not a clinician.
- These scripted decisions demonstrate workflow mechanics and FHIR state transitions.
- They are **not** a clinician usability evaluation, diagnostic validation, or real patient review.
- Public de-identified imaging remains linked only to synthetic FHIR R4 context.
- The low-confidence result is never finalized.
- Every decision stores reviewer role, timestamp, reason, note, and human-review Provenance.

In [1]:
# Cell 1 — Mount Drive and enforce the completed Notebook 07 gate

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import subprocess
import sys
import textwrap
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
NOTEBOOK_FILENAME = "08_NeuroFHIR_QC_Human_Review_Workflow.ipynb"
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

PROJECT_CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"

NB07_AUDIT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence_writeback_audit.json"
)
NB07_OUTPUT_ROOT = (
    PROJECT_ROOT / "submission/fhir_resources/notebook_07"
)
NB07_RESOURCE_ROOT = NB07_OUTPUT_ROOT / "resources"
NB07_EVIDENCE_MANIFEST_PATH = (
    NB07_OUTPUT_ROOT / "fhir_evidence_manifest.json"
)
NB07_TRANSACTION_REPORT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/"
      "transaction_writeback_report.json"
)
NB07_READBACK_REPORT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/"
      "readback_integrity_report.json"
)
SOURCE_RESOURCE_INDEX_PATH = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/resource_index.json"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "submission/fhir_resources/notebook_08"
)
REVIEW_RESOURCE_ROOT = OUTPUT_ROOT / "reviewed_resources"
REVIEW_SNAPSHOT_ROOT = OUTPUT_ROOT / "transition_snapshots"
REVIEW_BUNDLE_ROOT = OUTPUT_ROOT / "review_transaction_bundles"
SERVER_RESPONSE_ROOT = OUTPUT_ROOT / "server_responses"
READBACK_ROOT = OUTPUT_ROOT / "readback"
VALIDATION_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_08_human_review/validation"
)
EVAL_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_08_human_review"
)
DOC_ROOT = PROJECT_ROOT / "docs"

REVIEW_MANIFEST_PATH = OUTPUT_ROOT / "human_review_manifest.json"
MASTER_REVIEW_BUNDLE_PATH = (
    OUTPUT_ROOT / "master_review_evidence_bundle.json"
)
TRANSITION_REPORT_JSON = EVAL_ROOT / "review_transition_report.json"
TRANSITION_REPORT_CSV = EVAL_ROOT / "review_transition_report.csv"
LOCAL_VALIDATION_PATH = EVAL_ROOT / "local_workflow_validation.json"
SERVER_VALIDATION_PATH = EVAL_ROOT / "server_validation_report.json"
TRANSACTION_REPORT_PATH = EVAL_ROOT / "transaction_report.json"
READBACK_REPORT_PATH = EVAL_ROOT / "readback_integrity_report.json"
NETWORK_LOG_PATH = EVAL_ROOT / "network_request_log.json"
METRICS_PATH = EVAL_ROOT / "human_review_metrics.json"
AUDIT_JSON_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_08_human_review_workflow_audit.json"
)
AUDIT_MD_PATH = (
    DOC_ROOT / "NOTEBOOK_08_HUMAN_REVIEW_WORKFLOW.md"
)

REVIEW_SERVICE_PATH = (
    PROJECT_ROOT
    / "backend/app/services/human_review_service.py"
)
VERIFY_SCRIPT_PATH = (
    PROJECT_ROOT / "scripts/verify_human_review_workflow.py"
)
REQUIREMENTS_PATH = (
    PROJECT_ROOT / "requirements/human_review.txt"
)
TECH_DOC_PATH = (
    DOC_ROOT / "HUMAN_REVIEW_WORKFLOW.md"
)

for folder in (
    OUTPUT_ROOT,
    REVIEW_RESOURCE_ROOT,
    REVIEW_SNAPSHOT_ROOT,
    REVIEW_BUNDLE_ROOT,
    SERVER_RESPONSE_ROOT,
    READBACK_ROOT,
    VALIDATION_ROOT,
    EVAL_ROOT,
    DOC_ROOT,
    REVIEW_SERVICE_PATH.parent,
    VERIFY_SCRIPT_PATH.parent,
    REQUIREMENTS_PATH.parent,
):
    folder.mkdir(parents=True, exist_ok=True)

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")
    temporary.replace(path)

def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

required_paths = [
    NB07_AUDIT_PATH,
    NB07_EVIDENCE_MANIFEST_PATH,
    NB07_TRANSACTION_REPORT_PATH,
    NB07_READBACK_REPORT_PATH,
    SOURCE_RESOURCE_INDEX_PATH,
    NOTEBOOK_MANIFEST_PATH,
]
missing = [
    str(path)
    for path in required_paths
    if not path.exists() or path.stat().st_size == 0
]
if missing:
    raise FileNotFoundError(
        "Notebook 08 prerequisites are missing:\n"
        + "\n".join(f" - {path}" for path in missing)
    )

project_config = (
    load_json(PROJECT_CONFIG_PATH)
    if PROJECT_CONFIG_PATH.exists()
    else {"project_name": "NeuroFHIR-QC", "version": "0.1.0"}
)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
nb07_audit = load_json(NB07_AUDIT_PATH)
nb07_evidence_manifest = load_json(NB07_EVIDENCE_MANIFEST_PATH)
nb07_transaction = load_json(NB07_TRANSACTION_REPORT_PATH)
nb07_readback = load_json(NB07_READBACK_REPORT_PATH)
source_resource_index = load_json(SOURCE_RESOURCE_INDEX_PATH)

if nb07_audit.get("status") != "completed":
    raise RuntimeError("Notebook 07 audit status is not completed.")

metrics_07 = nb07_audit.get("metrics", {})
required_metric_values = {
    "case_count": 3,
    "unique_resource_count": 13,
    "transaction_bundle_count": 3,
    "local_validation_pass_rate": 1.0,
    "server_validation_pass_rate": 1.0,
    "transaction_success_rate": 1.0,
    "direct_read_success_rate": 1.0,
    "critical_field_preservation_rate": 1.0,
    "preliminary_observation_rate": 1.0,
    "preliminary_report_rate": 1.0,
    "review_task_requested_rate": 1.0,
    "autonomous_finalization_block_rate": 1.0,
}
for key, expected in required_metric_values.items():
    actual = metrics_07.get(key)
    if actual != expected:
        raise AssertionError(
            f"Notebook 07 gate failed: {key}={actual!r}, "
            f"expected {expected!r}."
        )

scope_07 = nb07_audit.get("scope", {})
if scope_07.get("human_review_transition_executed") is not False:
    raise AssertionError(
        "Notebook 07 must end before human-review transitions."
    )
if nb07_evidence_manifest.get("unique_resource_count") != 13:
    raise AssertionError("Notebook 07 evidence manifest is incomplete.")
if nb07_transaction.get("transaction_success_rate") != 1.0:
    raise AssertionError("Notebook 07 transactions did not fully pass.")
if nb07_readback.get("critical_field_preservation_rate") != 1.0:
    raise AssertionError("Notebook 07 read-back integrity did not fully pass.")

CASE_ORDER = ("stable", "progression", "low-confidence")
case_ids_07 = tuple(
    row["case_id"]
    for row in nb07_evidence_manifest.get("cases", [])
)
if set(case_ids_07) != set(CASE_ORDER):
    raise AssertionError(
        f"Notebook 07 does not contain the locked cases: {case_ids_07}"
    )

FHIR_BASE_URL = os.getenv(
    "NEUROFHIR_QC_NOTEBOOK08_FHIR_BASE_URL",
    nb07_audit.get("server_base_url", "https://hapi.fhir.org/baseR4"),
).rstrip("/")
FHIR_TIMEOUT_SECONDS = int(
    os.getenv("NEUROFHIR_QC_FHIR_TIMEOUT_SECONDS", "60")
)
ALLOW_SYNTHETIC_REVIEW_WRITEBACK = True

print("=" * 108)
print("✅ Notebook 07 completion gate passed")
print("✅ 13 preliminary/requested FHIR evidence resources are available")
print("✅ Three locked synthetic demonstration cases are present")
print(f"🌐 FHIR R4 server: {FHIR_BASE_URL}")
print("⚠️ Scripted synthetic research review — not clinician evaluation")
print("⚠️ No PHI or real patient identity may be used")
print("=" * 108)

Mounted at /content/drive
✅ Notebook 07 completion gate passed
✅ 13 preliminary/requested FHIR evidence resources are available
✅ Three locked synthetic demonstration cases are present
🌐 FHIR R4 server: https://hapi.fhir.org/baseR4
⚠️ Scripted synthetic research review — not clinician evaluation
⚠️ No PHI or real patient identity may be used


In [2]:
# Cell 2 — Load and revalidate the Notebook 07 evidence graph

def resource_reference(resource: dict[str, Any]) -> str:
    return f"{resource['resourceType']}/{resource['id']}"

def load_resource_by_reference(
    reference: str,
) -> dict[str, Any]:
    resource_type, resource_id = reference.split("/", 1)
    candidate = (
        NB07_RESOURCE_ROOT
        / f"{resource_type}-{resource_id}.json"
    )
    if not candidate.exists() or candidate.stat().st_size == 0:
        raise FileNotFoundError(candidate)
    resource = load_json(candidate)
    if resource_reference(resource) != reference:
        raise AssertionError(
            f"Resource/reference mismatch: {reference}"
        )
    return resource

device_reference = nb07_evidence_manifest["shared_device_reference"]
device_resource = load_resource_by_reference(device_reference)

case_source_resources: dict[str, list[dict[str, Any]]] = {}
case_evidence: dict[str, dict[str, dict[str, Any]]] = {}

for case_row in nb07_evidence_manifest["cases"]:
    case_id = case_row["case_id"]

    source_rows = [
        row
        for row in source_resource_index.get("resources", [])
        if row.get("case_id") == case_id
    ]
    if len(source_rows) != 5:
        raise AssertionError(
            f"{case_id}: expected 5 Notebook 01 source resources, "
            f"found {len(source_rows)}."
        )

    source_resources = []
    for row in source_rows:
        path = PROJECT_ROOT / row["relative_path"]
        if not path.exists() or path.stat().st_size == 0:
            raise FileNotFoundError(path)
        resource = load_json(path)
        if resource_reference(resource) != (
            f"{row['resource_type']}/{row['resource_id']}"
        ):
            raise AssertionError(
                f"{case_id}: source index mismatch for {path}"
            )
        source_resources.append(resource)
    case_source_resources[case_id] = source_resources

    resources = {
        resource_type: load_resource_by_reference(reference)
        for resource_type, reference
        in case_row["resources"].items()
    }
    expected_types = {
        "Observation",
        "DiagnosticReport",
        "Task",
        "Provenance",
    }
    if set(resources) != expected_types:
        raise AssertionError(
            f"{case_id}: unexpected Notebook 07 resources: "
            f"{sorted(resources)}"
        )

    observation = resources["Observation"]
    report = resources["DiagnosticReport"]
    task = resources["Task"]
    algorithmic_provenance = resources["Provenance"]

    if observation.get("status") != "preliminary":
        raise AssertionError(
            f"{case_id}: Observation is not preliminary."
        )
    if report.get("status") != "preliminary":
        raise AssertionError(
            f"{case_id}: DiagnosticReport is not preliminary."
        )
    if task.get("status") != "requested":
        raise AssertionError(
            f"{case_id}: Task is not requested."
        )
    if resource_reference(observation) not in {
        item.get("reference")
        for item in report.get("result", [])
    }:
        raise AssertionError(
            f"{case_id}: DiagnosticReport does not reference Observation."
        )
    if task.get("focus", {}).get("reference") != resource_reference(
        observation
    ):
        raise AssertionError(
            f"{case_id}: Task focus does not reference Observation."
        )
    if not {
        resource_reference(observation),
        resource_reference(report),
        resource_reference(task),
    }.issubset(
        {
            item.get("reference")
            for item in algorithmic_provenance.get("target", [])
        }
    ):
        raise AssertionError(
            f"{case_id}: algorithmic Provenance targets are incomplete."
        )

    case_evidence[case_id] = resources

print("=" * 108)
print("✅ Notebook 07 evidence reloaded and revalidated")
print("✅ 3 preliminary Observations")
print("✅ 3 preliminary DiagnosticReports")
print("✅ 3 requested Tasks")
print("✅ 3 algorithmic-generation Provenance resources")
print("✅ 15 source-context resources and 1 shared Device available")
print("=" * 108)

✅ Notebook 07 evidence reloaded and revalidated
✅ 3 preliminary Observations
✅ 3 preliminary DiagnosticReports
✅ 3 requested Tasks
✅ 3 algorithmic-generation Provenance resources
✅ 15 source-context resources and 1 shared Device available


In [3]:
# Cell 3 — Define the scripted review plan and FHIR R4 review builders

PROJECT_SYSTEM_ROOT = "https://neurofhir-qc.org/fhir"
IDENTIFIER_ROOT = f"{PROJECT_SYSTEM_ROOT}/identifier"
CODE_SYSTEM = f"{PROJECT_SYSTEM_ROOT}/CodeSystem/neurofhir-qc"
TAG_SYSTEM = f"{PROJECT_SYSTEM_ROOT}/CodeSystem/data-origin"
REVIEW_ACTIVITY_SYSTEM = (
    f"{PROJECT_SYSTEM_ROOT}/CodeSystem/review-activity"
)
REVIEW_DECISION_SYSTEM = (
    f"{PROJECT_SYSTEM_ROOT}/CodeSystem/review-decision"
)

FHIR_ID_PATTERN = re.compile(r"^[A-Za-z0-9\-\.]{1,64}$")
REVIEWER_ID = "nqc-synthetic-research-reviewer"
REVIEWER_REFERENCE = f"Practitioner/{REVIEWER_ID}"

def slug(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9\-\.]+", "-", value.strip())
    cleaned = re.sub(r"-+", "-", cleaned).strip("-")
    if not cleaned:
        raise ValueError("FHIR id cannot be empty.")
    if len(cleaned) > 64:
        cleaned = cleaned[:64].rstrip("-")
    if not FHIR_ID_PATTERN.fullmatch(cleaned):
        raise ValueError(f"Invalid FHIR id: {cleaned}")
    return cleaned

def concept(
    system: str,
    code: str,
    display: str,
) -> dict[str, Any]:
    return {
        "coding": [
            {
                "system": system,
                "code": code,
                "display": display,
            }
        ],
        "text": display,
    }

def identifier(
    resource_type: str,
    value: str,
) -> dict[str, str]:
    return {
        "system": f"{IDENTIFIER_ROOT}/{resource_type}",
        "value": value,
    }

def synthetic_meta(*tags: str) -> dict[str, Any]:
    return {
        "profile": [],
        "tag": [
            {
                "system": TAG_SYSTEM,
                "code": slug(tag),
                "display": tag.replace("-", " ").title(),
            }
            for tag in tags
        ],
    }

def set_review_tags(
    resource: dict[str, Any],
    *new_tags: str,
) -> None:
    meta = resource.setdefault("meta", {})
    existing_tags = meta.setdefault("tag", [])
    removable = {
        "preliminary",
        "human-review-required",
        "ai-result-preliminary",
        "human-review-queue",
        "review-not-yet-performed",
    }
    retained = [
        tag
        for tag in existing_tags
        if tag.get("code") not in removable
    ]
    existing_codes = {
        tag.get("code") for tag in retained
    }
    for tag in new_tags:
        code = slug(tag)
        if code not in existing_codes:
            retained.append(
                {
                    "system": TAG_SYSTEM,
                    "code": code,
                    "display": tag.replace("-", " ").title(),
                }
            )
            existing_codes.add(code)
    meta["tag"] = retained

def detached_copy(resource: dict[str, Any]) -> dict[str, Any]:
    return json.loads(json.dumps(resource))

def build_synthetic_reviewer() -> dict[str, Any]:
    return {
        "resourceType": "Practitioner",
        "id": REVIEWER_ID,
        "meta": synthetic_meta(
            "synthetic-reviewer",
            "research-example-use",
            "not-a-clinician-identity",
        ),
        "identifier": [
            identifier("Practitioner", REVIEWER_ID)
        ],
        "active": True,
        "name": [
            {
                "use": "usual",
                "text": "Synthetic Research Reviewer",
                "family": "Reviewer",
                "given": ["Synthetic", "Research"],
            }
        ],
        "communication": [
            {
                "coding": [
                    {
                        "system": "urn:ietf:bcp:47",
                        "code": "en",
                        "display": "English",
                    }
                ],
                "text": "English",
            }
        ],
    }

REVIEW_PLAN = [
    {
        "event_id": "stable-accepted",
        "case_id": "stable",
        "decision": "accepted",
        "reason": (
            "High-confidence segmentation and stable longitudinal "
            "behavior were reviewed and accepted for the scripted "
            "synthetic research demonstration."
        ),
        "note": (
            "Source links, model provenance, QC evidence, and the "
            "longitudinal calculation were reviewed. The result is "
            "finalized only within this synthetic demonstration."
        ),
        "observation_status": "final",
        "report_status": "final",
        "task_status": "completed",
        "task_business_status": "accepted",
        "final_event_for_case": True,
    },
    {
        "event_id": "progression-accepted",
        "case_id": "progression",
        "decision": "accepted",
        "reason": (
            "High-confidence segmentation and meaningful longitudinal "
            "increase were consistent with the executed progression "
            "evidence and were accepted for the scripted demonstration."
        ),
        "note": (
            "The reviewer confirmed the source-study link, model "
            "provenance, QC result, and reported volume change."
        ),
        "observation_status": "final",
        "report_status": "final",
        "task_status": "completed",
        "task_business_status": "accepted",
        "final_event_for_case": True,
    },
    {
        "event_id": "low-confidence-correction-required",
        "case_id": "low-confidence",
        "decision": "correction-required",
        "reason": (
            "Severe perturbation instability, a manual-review-required "
            "QC category, and withheld longitudinal interpretation "
            "require segmentation correction or reprocessing."
        ),
        "note": (
            "The original AI output remains preliminary and traceable. "
            "No corrected result has been supplied in this scripted step."
        ),
        "observation_status": "preliminary",
        "report_status": "preliminary",
        "task_status": "on-hold",
        "task_business_status": "correction-required",
        "final_event_for_case": False,
    },
    {
        "event_id": "low-confidence-rejected",
        "case_id": "low-confidence",
        "decision": "rejected",
        "reason": (
            "The original unstable AI output is rejected after the "
            "correction-required branch because no corrected output was "
            "introduced into this scripted demonstration."
        ),
        "note": (
            "The rejected result is retained for audit and marked "
            "entered-in-error. It is never presented as a final result."
        ),
        "observation_status": "entered-in-error",
        "report_status": "entered-in-error",
        "task_status": "rejected",
        "task_business_status": "rejected",
        "final_event_for_case": True,
    },
]

if [row["event_id"] for row in REVIEW_PLAN] != [
    "stable-accepted",
    "progression-accepted",
    "low-confidence-correction-required",
    "low-confidence-rejected",
]:
    raise AssertionError("Review-plan order changed unexpectedly.")

def build_reviewed_resources(
    plan: dict[str, Any],
    source_resources: dict[str, dict[str, Any]],
    reviewer: dict[str, Any],
    reviewed_utc: str,
    prior_review_provenance: dict[str, Any] | None = None,
) -> dict[str, dict[str, Any]]:
    case_id = plan["case_id"]
    decision = plan["decision"]

    observation = detached_copy(source_resources["Observation"])
    report = detached_copy(source_resources["DiagnosticReport"])
    task = detached_copy(source_resources["Task"])
    algorithmic_provenance = source_resources["Provenance"]

    observation["status"] = plan["observation_status"]
    observation["issued"] = reviewed_utc
    observation.setdefault("note", []).append(
        {
            "authorReference": {
                "reference": resource_reference(reviewer)
            },
            "time": reviewed_utc,
            "text": (
                f"Human-review decision: {decision}. "
                f"Reason: {plan['reason']} Note: {plan['note']}"
            ),
        }
    )
    set_review_tags(
        observation,
        "human-reviewed",
        f"review-{decision}",
        plan["observation_status"],
    )

    report["status"] = plan["report_status"]
    report["issued"] = reviewed_utc
    report["resultsInterpreter"] = [
        {"reference": resource_reference(reviewer)}
    ]
    original_conclusion = report.get("conclusion", "").rstrip()
    report["conclusion"] = (
        f"{original_conclusion} Human-review decision: {decision}. "
        f"{plan['note']}"
    ).strip()
    set_review_tags(
        report,
        "human-reviewed",
        f"review-{decision}",
        plan["report_status"],
    )

    task["status"] = plan["task_status"]
    task["businessStatus"] = concept(
        REVIEW_DECISION_SYSTEM,
        plan["task_business_status"],
        plan["task_business_status"].replace("-", " ").title(),
    )
    task["lastModified"] = reviewed_utc
    task["owner"] = {
        "reference": resource_reference(reviewer)
    }
    task["executionPeriod"] = {
        "start": task.get("authoredOn", reviewed_utc),
        "end": reviewed_utc,
    }
    task.setdefault("note", []).append(
        {
            "authorReference": {
                "reference": resource_reference(reviewer)
            },
            "time": reviewed_utc,
            "text": (
                f"Review decision: {decision}. "
                f"Reason: {plan['reason']} Note: {plan['note']}"
            ),
        }
    )
    task.setdefault("output", []).extend(
        [
            {
                "type": concept(
                    REVIEW_DECISION_SYSTEM,
                    "review-decision",
                    "Review decision",
                ),
                "valueCodeableConcept": concept(
                    REVIEW_DECISION_SYSTEM,
                    decision,
                    decision.replace("-", " ").title(),
                ),
            },
            {
                "type": concept(
                    REVIEW_DECISION_SYSTEM,
                    "reviewer-role",
                    "Reviewer role",
                ),
                "valueString": "Synthetic research reviewer",
            },
            {
                "type": concept(
                    REVIEW_DECISION_SYSTEM,
                    "review-reason",
                    "Review reason",
                ),
                "valueString": plan["reason"],
            },
            {
                "type": concept(
                    REVIEW_DECISION_SYSTEM,
                    "review-note",
                    "Review note",
                ),
                "valueString": plan["note"],
            },
        ]
    )
    set_review_tags(
        task,
        "human-reviewed",
        f"review-{decision}",
        plan["task_status"],
    )

    provenance_id = slug(
        f"nqc-{case_id}-review-{decision}-provenance"
    )
    provenance_entities = [
        {
            "role": "revision",
            "what": {
                "reference": resource_reference(
                    algorithmic_provenance
                )
            },
        }
    ]
    if prior_review_provenance is not None:
        provenance_entities.append(
            {
                "role": "revision",
                "what": {
                    "reference": resource_reference(
                        prior_review_provenance
                    )
                },
            }
        )

    review_provenance = {
        "resourceType": "Provenance",
        "id": provenance_id,
        "meta": synthetic_meta(
            "human-review-event",
            f"review-{decision}",
            "synthetic-reviewer",
        ),
        "target": [
            {"reference": resource_reference(observation)},
            {"reference": resource_reference(report)},
            {"reference": resource_reference(task)},
        ],
        "occurredDateTime": reviewed_utc,
        "recorded": reviewed_utc,
        "reason": [
            concept(
                REVIEW_DECISION_SYSTEM,
                decision,
                decision.replace("-", " ").title(),
            )
        ],
        "activity": concept(
            REVIEW_ACTIVITY_SYSTEM,
            f"human-review-{decision}",
            f"Human review: {decision.replace('-', ' ')}",
        ),
        "agent": [
            {
                "type": concept(
                    "http://terminology.hl7.org/CodeSystem/"
                    "provenance-participant-type",
                    "verifier",
                    "Verifier",
                ),
                "role": [
                    concept(
                        REVIEW_DECISION_SYSTEM,
                        "synthetic-research-reviewer",
                        "Synthetic research reviewer",
                    )
                ],
                "who": {
                    "reference": resource_reference(reviewer)
                },
            }
        ],
        "entity": provenance_entities,
        "policy": [
            "https://neurofhir-qc.org/policy/"
            "scripted-synthetic-human-review"
        ],
    }

    return {
        "Observation": observation,
        "DiagnosticReport": report,
        "Task": task,
        "ReviewProvenance": review_provenance,
    }

print("=" * 108)
print("✅ Four-event scripted review plan defined")
print("✅ Stable and progression: accepted")
print("✅ Low confidence: correction-required, then rejected")
print("✅ Synthetic reviewer identity is explicit and non-clinical")
print("=" * 108)

✅ Four-event scripted review plan defined
✅ Stable and progression: accepted
✅ Low confidence: correction-required, then rejected
✅ Synthetic reviewer identity is explicit and non-clinical


In [4]:
# Cell 4 — Build transition snapshots and self-contained transaction Bundles

reviewed_utc = utc_now()
synthetic_reviewer = build_synthetic_reviewer()

review_event_outputs: dict[str, dict[str, dict[str, Any]]] = {}
review_provenance_by_event: dict[str, dict[str, Any]] = {}
final_case_resources: dict[str, dict[str, dict[str, Any]]] = {}
transition_rows: list[dict[str, Any]] = []

prior_low_confidence_provenance = None

for sequence, plan in enumerate(REVIEW_PLAN, start=1):
    case_id = plan["case_id"]
    source_resources = case_evidence[case_id]

    if plan["event_id"] == "low-confidence-rejected":
        source_resources = {
            **source_resources,
            "Observation": detached_copy(
                review_event_outputs[
                    "low-confidence-correction-required"
                ]["Observation"]
            ),
            "DiagnosticReport": detached_copy(
                review_event_outputs[
                    "low-confidence-correction-required"
                ]["DiagnosticReport"]
            ),
            "Task": detached_copy(
                review_event_outputs[
                    "low-confidence-correction-required"
                ]["Task"]
            ),
        }
        prior_low_confidence_provenance = (
            review_provenance_by_event[
                "low-confidence-correction-required"
            ]
        )

    before_statuses = {
        "Observation": source_resources["Observation"]["status"],
        "DiagnosticReport": source_resources[
            "DiagnosticReport"
        ]["status"],
        "Task": source_resources["Task"]["status"],
    }

    outputs = build_reviewed_resources(
        plan=plan,
        source_resources=source_resources,
        reviewer=synthetic_reviewer,
        reviewed_utc=reviewed_utc,
        prior_review_provenance=prior_low_confidence_provenance,
    )
    review_event_outputs[plan["event_id"]] = outputs
    review_provenance_by_event[plan["event_id"]] = outputs[
        "ReviewProvenance"
    ]

    after_statuses = {
        "Observation": outputs["Observation"]["status"],
        "DiagnosticReport": outputs[
            "DiagnosticReport"
        ]["status"],
        "Task": outputs["Task"]["status"],
    }

    transition_rows.append(
        {
            "sequence": sequence,
            "event_id": plan["event_id"],
            "case_id": case_id,
            "decision": plan["decision"],
            "reviewer_reference": REVIEWER_REFERENCE,
            "reviewer_role": "Synthetic research reviewer",
            "reviewed_utc": reviewed_utc,
            "reason": plan["reason"],
            "note": plan["note"],
            "before_observation_status": before_statuses[
                "Observation"
            ],
            "after_observation_status": after_statuses[
                "Observation"
            ],
            "before_report_status": before_statuses[
                "DiagnosticReport"
            ],
            "after_report_status": after_statuses[
                "DiagnosticReport"
            ],
            "before_task_status": before_statuses["Task"],
            "after_task_status": after_statuses["Task"],
            "review_provenance_reference": resource_reference(
                outputs["ReviewProvenance"]
            ),
            "final_event_for_case": plan["final_event_for_case"],
        }
    )

    event_dir = REVIEW_SNAPSHOT_ROOT / plan["event_id"]
    event_dir.mkdir(parents=True, exist_ok=True)
    for label, resource in outputs.items():
        write_json(
            event_dir
            / f"{resource['resourceType']}-{resource['id']}.json",
            resource,
        )

    if plan["final_event_for_case"]:
        final_case_resources[case_id] = outputs

if set(final_case_resources) != set(CASE_ORDER):
    raise AssertionError("Final reviewed case resources are incomplete.")

def deterministic_full_url(
    resource_type: str,
    resource_id: str,
) -> str:
    value = uuid.uuid5(
        uuid.NAMESPACE_URL,
        (
            "https://neurofhir-qc.org/fhir/"
            f"{resource_type}/{resource_id}"
        ),
    )
    return f"urn:uuid:{value}"

def collect_references(
    value: Any,
    *,
    path: str = "$",
) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}"
            if key == "reference" and isinstance(child, str):
                rows.append(
                    {
                        "path": child_path,
                        "reference": child,
                    }
                )
            else:
                rows.extend(
                    collect_references(child, path=child_path)
                )
    elif isinstance(value, list):
        for index, child in enumerate(value):
            rows.extend(
                collect_references(
                    child,
                    path=f"{path}[{index}]",
                )
            )
    return rows

def replace_bundle_references(
    value: Any,
    full_url_by_reference: dict[str, str],
) -> Any:
    if isinstance(value, dict):
        result: dict[str, Any] = {}
        for key, child in value.items():
            if (
                key == "reference"
                and isinstance(child, str)
                and child in full_url_by_reference
            ):
                result[key] = full_url_by_reference[child]
            else:
                result[key] = replace_bundle_references(
                    child,
                    full_url_by_reference,
                )
        return result
    if isinstance(value, list):
        return [
            replace_bundle_references(
                child,
                full_url_by_reference,
            )
            for child in value
        ]
    return value

def build_transaction_bundle(
    event_id: str,
    resources: Iterable[dict[str, Any]],
    timestamp: str,
) -> dict[str, Any]:
    resource_list = list(resources)
    references = [
        resource_reference(resource)
        for resource in resource_list
    ]
    if len(references) != len(set(references)):
        duplicates = sorted(
            ref
            for ref in set(references)
            if references.count(ref) > 1
        )
        raise AssertionError(
            f"{event_id}: duplicate Bundle resources: {duplicates}"
        )

    full_url_by_reference = {
        resource_reference(resource): deterministic_full_url(
            resource["resourceType"],
            resource["id"],
        )
        for resource in resource_list
    }

    entries = []
    for resource in resource_list:
        ref = resource_reference(resource)
        bundled_resource = detached_copy(resource)
        bundled_resource = replace_bundle_references(
            bundled_resource,
            full_url_by_reference,
        )
        entries.append(
            {
                "fullUrl": full_url_by_reference[ref],
                "resource": bundled_resource,
                "request": {
                    "method": "PUT",
                    "url": ref,
                },
            }
        )

    bundle_id = slug(f"nqc-{event_id}-review-transaction")
    return {
        "resourceType": "Bundle",
        "id": bundle_id,
        "meta": synthetic_meta(
            "human-review-transaction",
            "synthetic-reviewer",
        ),
        "identifier": {
            "system": f"{IDENTIFIER_ROOT}/Bundle",
            "value": bundle_id,
        },
        "type": "transaction",
        "timestamp": timestamp,
        "entry": entries,
    }

event_bundles: dict[str, dict[str, Any]] = {}

for plan in REVIEW_PLAN:
    event_id = plan["event_id"]
    case_id = plan["case_id"]
    outputs = review_event_outputs[event_id]

    bundle_resources = [
        *case_source_resources[case_id],
        device_resource,
        outputs["Observation"],
        outputs["DiagnosticReport"],
        outputs["Task"],
        case_evidence[case_id]["Provenance"],
        synthetic_reviewer,
    ]

    if event_id == "low-confidence-rejected":
        bundle_resources.append(
            review_provenance_by_event[
                "low-confidence-correction-required"
            ]
        )

    bundle_resources.append(outputs["ReviewProvenance"])

    expected_count = (
        13 if event_id == "low-confidence-rejected" else 12
    )
    if len(bundle_resources) != expected_count:
        raise AssertionError(
            f"{event_id}: expected {expected_count} resources, "
            f"found {len(bundle_resources)}."
        )

    bundle = build_transaction_bundle(
        event_id,
        bundle_resources,
        reviewed_utc,
    )

    bundle_full_urls = {
        entry["fullUrl"] for entry in bundle["entry"]
    }
    bundle_relative_references = {
        resource_reference(resource)
        for resource in bundle_resources
    }
    unresolved = []
    for entry in bundle["entry"]:
        for item in collect_references(entry["resource"]):
            target = item["reference"]
            if target in bundle_relative_references:
                unresolved.append(
                    {
                        "source": resource_reference(
                            entry["resource"]
                        ),
                        "path": item["path"],
                        "target": target,
                    }
                )
            if (
                target.startswith("urn:uuid:")
                and target not in bundle_full_urls
            ):
                unresolved.append(
                    {
                        "source": resource_reference(
                            entry["resource"]
                        ),
                        "path": item["path"],
                        "target": target,
                    }
                )
    if unresolved:
        raise AssertionError(
            f"{event_id}: Bundle URN rewriting failed:\n"
            + json.dumps(unresolved, indent=2)
        )

    event_bundles[event_id] = bundle
    write_json(
        REVIEW_BUNDLE_ROOT / f"{event_id}_transaction_bundle.json",
        bundle,
    )

for stale in REVIEW_RESOURCE_ROOT.glob("*.json"):
    stale.unlink()

review_unique_resources: dict[str, dict[str, Any]] = {
    resource_reference(synthetic_reviewer): synthetic_reviewer,
}
for case_id, outputs in final_case_resources.items():
    for label in (
        "Observation",
        "DiagnosticReport",
        "Task",
        "ReviewProvenance",
    ):
        resource = outputs[label]
        review_unique_resources[
            resource_reference(resource)
        ] = resource

correction_provenance = review_provenance_by_event[
    "low-confidence-correction-required"
]
review_unique_resources[
    resource_reference(correction_provenance)
] = correction_provenance

if len(review_unique_resources) != 14:
    raise AssertionError(
        f"Expected 14 unique human-review resources, "
        f"found {len(review_unique_resources)}."
    )

for reference, resource in sorted(review_unique_resources.items()):
    write_json(
        REVIEW_RESOURCE_ROOT
        / f"{resource['resourceType']}-{resource['id']}.json",
        resource,
    )

master_review_bundle = {
    "resourceType": "Bundle",
    "id": "nqc-notebook08-master-review-evidence",
    "meta": synthetic_meta(
        "human-review-evidence-collection",
        "not-for-transaction",
    ),
    "identifier": {
        "system": f"{IDENTIFIER_ROOT}/Bundle",
        "value": "nqc-notebook08-master-review-evidence",
    },
    "type": "collection",
    "timestamp": reviewed_utc,
    "entry": [
        {
            "fullUrl": deterministic_full_url(
                resource["resourceType"],
                resource["id"],
            ),
            "resource": resource,
        }
        for resource in review_unique_resources.values()
    ],
}
write_json(MASTER_REVIEW_BUNDLE_PATH, master_review_bundle)

review_manifest = {
    "project_name": project_config.get(
        "project_name",
        "NeuroFHIR-QC",
    ),
    "notebook_number": "08",
    "generated_utc": reviewed_utc,
    "fhir_version": "4.0.1",
    "reviewer": {
        "reference": REVIEWER_REFERENCE,
        "display": "Synthetic Research Reviewer",
        "role": "Synthetic research reviewer",
        "real_clinician": False,
        "usability_evaluator": False,
    },
    "transition_event_count": 4,
    "final_case_count": 3,
    "unique_review_resource_count": 14,
    "transaction_bundle_count": 4,
    "events": transition_rows,
    "final_case_resources": {
        case_id: {
            label: resource_reference(resource)
            for label, resource in outputs.items()
        }
        for case_id, outputs in final_case_resources.items()
    },
    "correction_required_provenance": resource_reference(
        correction_provenance
    ),
    "master_review_bundle": MASTER_REVIEW_BUNDLE_PATH.relative_to(
        PROJECT_ROOT
    ).as_posix(),
}
write_json(REVIEW_MANIFEST_PATH, review_manifest)

print("=" * 108)
print("✅ Four review transition events generated")
print("✅ 14 unique human-review FHIR resources generated")
print("✅ 4 self-contained review transaction Bundles generated")
print("✅ Low-confidence correction-required snapshot preserved")
print("✅ Final low-confidence state is rejected/entered-in-error")
print("=" * 108)

✅ Four review transition events generated
✅ 14 unique human-review FHIR resources generated
✅ 4 self-contained review transaction Bundles generated
✅ Low-confidence correction-required snapshot preserved
✅ Final low-confidence state is rejected/entered-in-error


In [5]:
# Cell 5 — Run local FHIR, status-transition, safety, and reference validation

allowed_status_transitions = {
    ("preliminary", "final"),
    ("preliminary", "preliminary"),
    ("preliminary", "entered-in-error"),
    ("requested", "completed"),
    ("requested", "on-hold"),
    ("on-hold", "rejected"),
}

transition_errors: list[str] = []

for row in transition_rows:
    observation_transition = (
        row["before_observation_status"],
        row["after_observation_status"],
    )
    report_transition = (
        row["before_report_status"],
        row["after_report_status"],
    )
    task_transition = (
        row["before_task_status"],
        row["after_task_status"],
    )

    if observation_transition not in allowed_status_transitions:
        transition_errors.append(
            f"{row['event_id']}: invalid Observation transition "
            f"{observation_transition}"
        )
    if report_transition not in allowed_status_transitions:
        transition_errors.append(
            f"{row['event_id']}: invalid DiagnosticReport transition "
            f"{report_transition}"
        )
    if task_transition not in allowed_status_transitions:
        transition_errors.append(
            f"{row['event_id']}: invalid Task transition "
            f"{task_transition}"
        )

if transition_errors:
    raise AssertionError(
        "Invalid review transitions:\n"
        + "\n".join(f" - {error}" for error in transition_errors)
    )

expected_final_statuses = {
    "stable": {
        "Observation": "final",
        "DiagnosticReport": "final",
        "Task": "completed",
    },
    "progression": {
        "Observation": "final",
        "DiagnosticReport": "final",
        "Task": "completed",
    },
    "low-confidence": {
        "Observation": "entered-in-error",
        "DiagnosticReport": "entered-in-error",
        "Task": "rejected",
    },
}

for case_id, expected in expected_final_statuses.items():
    outputs = final_case_resources[case_id]
    for resource_type, expected_status in expected.items():
        actual = outputs[resource_type]["status"]
        if actual != expected_status:
            raise AssertionError(
                f"{case_id}: {resource_type}.status={actual}, "
                f"expected {expected_status}."
            )

correction_outputs = review_event_outputs[
    "low-confidence-correction-required"
]
if correction_outputs["Observation"]["status"] != "preliminary":
    raise AssertionError(
        "Correction-required Observation must remain preliminary."
    )
if correction_outputs["DiagnosticReport"]["status"] != "preliminary":
    raise AssertionError(
        "Correction-required DiagnosticReport must remain preliminary."
    )
if correction_outputs["Task"]["status"] != "on-hold":
    raise AssertionError(
        "Correction-required Task must be on-hold."
    )

if synthetic_reviewer["name"][0]["text"] != (
    "Synthetic Research Reviewer"
):
    raise AssertionError("Synthetic reviewer labeling changed.")
if any(
    "clinician" in json.dumps(resource).lower()
    and "not-a-clinician-identity" not in json.dumps(resource).lower()
    for resource in review_unique_resources.values()
):
    raise AssertionError(
        "Unexpected clinician identity or claim detected."
    )

required_fields = {
    "Practitioner": {"resourceType", "id", "active", "name"},
    "Observation": {
        "resourceType",
        "id",
        "status",
        "code",
        "subject",
        "valueQuantity",
    },
    "DiagnosticReport": {
        "resourceType",
        "id",
        "status",
        "code",
        "subject",
        "result",
    },
    "Task": {
        "resourceType",
        "id",
        "status",
        "intent",
        "focus",
        "for",
        "owner",
        "output",
    },
    "Provenance": {
        "resourceType",
        "id",
        "target",
        "recorded",
        "activity",
        "agent",
    },
}

resource_validation_rows = []
for reference, resource in sorted(review_unique_resources.items()):
    resource_type = resource["resourceType"]
    missing_fields = sorted(
        required_fields[resource_type] - set(resource)
    )
    valid_id = bool(FHIR_ID_PATTERN.fullmatch(resource["id"]))
    passed = not missing_fields and valid_id
    resource_validation_rows.append(
        {
            "reference": reference,
            "resource_type": resource_type,
            "missing_required_fields": missing_fields,
            "valid_fhir_id": valid_id,
            "passed": passed,
        }
    )

failed_resources = [
    row for row in resource_validation_rows
    if not row["passed"]
]
if failed_resources:
    raise AssertionError(
        "Local FHIR structure validation failed:\n"
        + json.dumps(failed_resources, indent=2)
    )

local_reference_universe = {
    resource_reference(resource)
    for resources in case_source_resources.values()
    for resource in resources
}
local_reference_universe.update(
    {
        resource_reference(device_resource),
        *[
            resource_reference(
                case_evidence[case_id]["Provenance"]
            )
            for case_id in CASE_ORDER
        ],
        *review_unique_resources.keys(),
    }
)

reference_rows = []
for source_ref, resource in sorted(
    review_unique_resources.items()
):
    for item in collect_references(resource):
        target = item["reference"]
        relative_reference = (
            "/" in target
            and not target.startswith(("http://", "https://"))
        )
        resolved = (
            target in local_reference_universe
            if relative_reference
            else True
        )
        reference_rows.append(
            {
                "source_reference": source_ref,
                "path": item["path"],
                "target_reference": target,
                "resolved": resolved,
            }
        )

unresolved_references = [
    row for row in reference_rows
    if not row["resolved"]
]
if unresolved_references:
    raise AssertionError(
        "Local review reference validation failed:\n"
        + json.dumps(unresolved_references, indent=2)
    )

for event_id, bundle in event_bundles.items():
    if bundle.get("type") != "transaction":
        raise AssertionError(
            f"{event_id}: Bundle is not a transaction."
        )
    if any(
        entry.get("request", {}).get("method") != "PUT"
        for entry in bundle.get("entry", [])
    ):
        raise AssertionError(
            f"{event_id}: non-PUT transaction entry found."
        )

local_validation = {
    "project_name": project_config.get(
        "project_name",
        "NeuroFHIR-QC",
    ),
    "validated_utc": utc_now(),
    "transition_event_count": len(transition_rows),
    "transition_validation_pass_rate": 1.0,
    "unique_review_resource_count": len(
        review_unique_resources
    ),
    "resource_structure_pass_rate": 1.0,
    "reference_count": len(reference_rows),
    "reference_integrity_rate": 1.0,
    "transaction_bundle_count": len(event_bundles),
    "synthetic_reviewer_explicit": True,
    "real_clinician_identity_used": False,
    "low_confidence_finalized": False,
    "transition_rows": transition_rows,
    "resource_rows": resource_validation_rows,
    "reference_rows": reference_rows,
}
write_json(LOCAL_VALIDATION_PATH, local_validation)

with TRANSITION_REPORT_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(transition_rows[0].keys()),
    )
    writer.writeheader()
    writer.writerows(transition_rows)

write_json(
    TRANSITION_REPORT_JSON,
    {
        "generated_utc": reviewed_utc,
        "event_count": len(transition_rows),
        "events": transition_rows,
    },
)

print("=" * 108)
print("✅ Local review workflow validation passed")
print("✅ 4/4 status transitions valid")
print("✅ 14/14 review resources structurally valid")
print(
    f"✅ {len(reference_rows)}/{len(reference_rows)} "
    "local review references resolved"
)
print("✅ Low-confidence result was never finalized")
print("=" * 108)

✅ Local review workflow validation passed
✅ 4/4 status transitions valid
✅ 14/14 review resources structurally valid
✅ 68/68 local review references resolved
✅ Low-confidence result was never finalized


In [6]:
# Cell 6 — Configure the FHIR R4 client and verify server capabilities

packages = [
    "requests>=2.32,<3",
    "urllib3>=2,<3",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages]
)

import requests
import urllib3

runtime_versions = {
    "python": sys.version.split()[0],
    "requests": requests.__version__,
    "urllib3": urllib3.__version__,
}

session = requests.Session()
session.headers.update(
    {
        "Accept": "application/fhir+json, application/json",
        "Content-Type": "application/fhir+json",
        "User-Agent": "NeuroFHIR-QC-Notebook08-HumanReview",
    }
)

network_log: list[dict[str, Any]] = []

def request_fhir(
    method: str,
    endpoint: str,
    *,
    payload: dict[str, Any] | None = None,
    purpose: str,
    retry_read_like: bool,
    max_attempts: int = 4,
) -> requests.Response:
    url = (
        FHIR_BASE_URL
        if endpoint == ""
        else f"{FHIR_BASE_URL}/{endpoint.lstrip('/')}"
    )
    attempts = max_attempts if retry_read_like else 1
    last_exception: Exception | None = None

    for attempt in range(1, attempts + 1):
        started = time.perf_counter()
        try:
            response = session.request(
                method=method,
                url=url,
                json=payload,
                timeout=FHIR_TIMEOUT_SECONDS,
            )
            elapsed = time.perf_counter() - started
            network_log.append(
                {
                    "timestamp_utc": utc_now(),
                    "purpose": purpose,
                    "method": method,
                    "url": url,
                    "attempt": attempt,
                    "status_code": response.status_code,
                    "elapsed_seconds": round(elapsed, 6),
                }
            )

            transient = response.status_code in {
                429,
                500,
                502,
                503,
                504,
            }
            if (
                retry_read_like
                and transient
                and attempt < attempts
            ):
                time.sleep(min(2 ** (attempt - 1), 8))
                continue
            return response
        except requests.RequestException as exc:
            elapsed = time.perf_counter() - started
            last_exception = exc
            network_log.append(
                {
                    "timestamp_utc": utc_now(),
                    "purpose": purpose,
                    "method": method,
                    "url": url,
                    "attempt": attempt,
                    "status_code": None,
                    "elapsed_seconds": round(elapsed, 6),
                    "exception": repr(exc),
                }
            )
            if not retry_read_like or attempt >= attempts:
                raise
            time.sleep(min(2 ** (attempt - 1), 8))

    raise RuntimeError(
        f"FHIR request failed: {purpose}"
    ) from last_exception

metadata_response = request_fhir(
    "GET",
    "metadata",
    purpose="Read FHIR CapabilityStatement",
    retry_read_like=True,
)
if not metadata_response.ok:
    raise RuntimeError(
        "CapabilityStatement retrieval failed: "
        f"{metadata_response.status_code} "
        f"{metadata_response.text[:1000]}"
    )

capability_statement = metadata_response.json()
if capability_statement.get("resourceType") != (
    "CapabilityStatement"
):
    raise AssertionError(
        "Server metadata is not a CapabilityStatement."
    )

fhir_version = capability_statement.get("fhirVersion")
if not str(fhir_version).startswith("4.0"):
    raise AssertionError(
        f"Expected FHIR R4 server, received {fhir_version!r}."
    )

advertised_resources = {
    item.get("type")
    for rest in capability_statement.get("rest", [])
    for item in rest.get("resource", [])
}
required_resource_types = {
    "Bundle",
    "Practitioner",
    "Observation",
    "DiagnosticReport",
    "Task",
    "Provenance",
}
missing_advertised = sorted(
    required_resource_types - advertised_resources
)
if missing_advertised:
    raise AssertionError(
        "FHIR server does not advertise required resources: "
        + ", ".join(missing_advertised)
    )

write_json(
    EVAL_ROOT / "capability_statement.json",
    capability_statement,
)

def operation_outcome_summary(
    payload: Any,
) -> list[dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    if payload.get("resourceType") != "OperationOutcome":
        return []
    rows = []
    for issue in payload.get("issue", []):
        rows.append(
            {
                "severity": issue.get("severity"),
                "code": issue.get("code"),
                "diagnostics": issue.get("diagnostics"),
                "details": issue.get("details", {}).get("text"),
                "expression": issue.get("expression", []),
                "location": issue.get("location", []),
            }
        )
    return rows

def validation_passed(
    response: requests.Response,
    payload: Any,
) -> tuple[bool, list[dict[str, Any]]]:
    issues = operation_outcome_summary(payload)
    blocking = [
        issue
        for issue in issues
        if issue.get("severity") in {"fatal", "error"}
    ]
    return response.ok and not blocking, issues

def blocking_issue_text(
    label: str,
    issues: list[dict[str, Any]],
) -> str:
    blocking = [
        issue
        for issue in issues
        if issue.get("severity") in {"fatal", "error"}
    ]
    lines = [f"{label}:"]
    for issue in blocking:
        lines.append(
            "  - "
            + str(
                issue.get("diagnostics")
                or issue.get("details")
                or issue.get("code")
            )
            + (
                f" | expression={issue.get('expression')}"
                if issue.get("expression")
                else ""
            )
        )
    return "\n".join(lines)

print("=" * 108)
print("✅ FHIR server CapabilityStatement verified")
print(f"✅ FHIR version: {fhir_version}")
print(
    "✅ Required review resource types advertised: "
    + ", ".join(sorted(required_resource_types))
)
print("=" * 108)

✅ FHIR server CapabilityStatement verified
✅ FHIR version: 4.0.1
✅ Required review resource types advertised: Bundle, DiagnosticReport, Observation, Practitioner, Provenance, Task


In [7]:
# Cell 7 — Validate, write, and read back the four review transitions

if not ALLOW_SYNTHETIC_REVIEW_WRITEBACK:
    raise RuntimeError(
        "Synthetic review write-back is disabled."
    )

for folder in (
    VALIDATION_ROOT,
    SERVER_RESPONSE_ROOT,
    READBACK_ROOT,
):
    for stale_file in folder.glob("*.json"):
        stale_file.unlink()

bundle_validation_rows = []
bundle_validation_issues: dict[str, list[dict[str, Any]]] = {}

for sequence, plan in enumerate(REVIEW_PLAN, start=1):
    event_id = plan["event_id"]
    bundle = event_bundles[event_id]
    response = request_fhir(
        "POST",
        "Bundle/$validate",
        payload=bundle,
        purpose=f"Validate review transaction Bundle: {event_id}",
        retry_read_like=True,
    )
    try:
        payload = response.json()
    except Exception:
        payload = {
            "resourceType": "OperationOutcome",
            "issue": [
                {
                    "severity": "error",
                    "code": "exception",
                    "diagnostics": response.text[:1000],
                }
            ],
        }

    passed, issues = validation_passed(response, payload)
    bundle_validation_issues[event_id] = issues
    outcome_path = (
        VALIDATION_ROOT
        / f"{sequence:02d}_{event_id}_bundle_operation_outcome.json"
    )
    write_json(outcome_path, payload)

    bundle_validation_rows.append(
        {
            "phase": "prewrite-bundle",
            "event_id": event_id,
            "status_code": response.status_code,
            "entry_count": len(bundle.get("entry", [])),
            "passed": passed,
            "blocking_issue_count": sum(
                issue.get("severity") in {"fatal", "error"}
                for issue in issues
            ),
            "warning_issue_count": sum(
                issue.get("severity") == "warning"
                for issue in issues
            ),
            "outcome_file": outcome_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
    )

failed_bundle_validations = [
    row for row in bundle_validation_rows
    if not row["passed"]
]
if failed_bundle_validations:
    details = [
        blocking_issue_text(
            row["event_id"],
            bundle_validation_issues[row["event_id"]],
        )
        for row in failed_bundle_validations
    ]
    raise AssertionError(
        "Review Bundle validation failed:\n"
        + "\n".join(details)
    )

transaction_rows = []
transaction_entry_rows = []
intermediate_correction_readback = {}

for sequence, plan in enumerate(REVIEW_PLAN, start=1):
    event_id = plan["event_id"]
    bundle = event_bundles[event_id]

    response = request_fhir(
        "POST",
        "",
        payload=bundle,
        purpose=f"Write review transaction: {event_id}",
        retry_read_like=False,
    )
    try:
        response_payload = response.json()
    except Exception as exc:
        raise RuntimeError(
            f"{event_id}: transaction returned non-JSON: "
            f"{response.text[:1000]}"
        ) from exc

    response_path = (
        SERVER_RESPONSE_ROOT
        / f"{sequence:02d}_{event_id}_transaction_response.json"
    )
    write_json(response_path, response_payload)

    if response_payload.get("resourceType") != "Bundle":
        raise AssertionError(
            f"{event_id}: response is not a Bundle."
        )
    if response_payload.get("type") != "transaction-response":
        raise AssertionError(
            f"{event_id}: response is not transaction-response."
        )

    response_entries = response_payload.get("entry", [])
    expected_entries = len(bundle.get("entry", []))
    if len(response_entries) != expected_entries:
        raise AssertionError(
            f"{event_id}: response entry count mismatch."
        )

    success_count = 0
    for index, response_entry in enumerate(response_entries):
        response_detail = response_entry.get("response", {})
        status_text = str(response_detail.get("status", ""))
        match = re.match(r"^(\d{3})", status_text)
        status_code = int(match.group(1)) if match else 0
        passed = 200 <= status_code < 300
        success_count += int(passed)
        transaction_entry_rows.append(
            {
                "event_id": event_id,
                "entry_index": index,
                "request_url": bundle["entry"][index][
                    "request"
                ]["url"],
                "response_status": status_text,
                "passed": passed,
            }
        )

    transaction_passed = (
        response.ok and success_count == expected_entries
    )
    transaction_rows.append(
        {
            "event_id": event_id,
            "case_id": plan["case_id"],
            "decision": plan["decision"],
            "http_status_code": response.status_code,
            "entry_count": expected_entries,
            "successful_entry_count": success_count,
            "transaction_passed": transaction_passed,
            "response_file": response_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
    )

    if not transaction_passed:
        raise AssertionError(
            f"{event_id}: transaction failed "
            f"({success_count}/{expected_entries} entries)."
        )

    if event_id == "low-confidence-correction-required":
        correction_outputs = review_event_outputs[event_id]
        correction_targets = {
            resource_reference(correction_outputs["Observation"]):
                "preliminary",
            resource_reference(
                correction_outputs["DiagnosticReport"]
            ): "preliminary",
            resource_reference(correction_outputs["Task"]):
                "on-hold",
            resource_reference(
                correction_outputs["ReviewProvenance"]
            ): None,
        }

        for reference, expected_status in (
            correction_targets.items()
        ):
            read_response = request_fhir(
                "GET",
                reference,
                purpose=(
                    "Read correction-required intermediate state: "
                    + reference
                ),
                retry_read_like=True,
            )
            if not read_response.ok:
                raise RuntimeError(
                    f"Correction intermediate read failed: "
                    f"{reference} HTTP {read_response.status_code}"
                )
            server_resource = read_response.json()
            if (
                expected_status is not None
                and server_resource.get("status")
                != expected_status
            ):
                raise AssertionError(
                    f"{reference}: correction-required read-back "
                    f"status={server_resource.get('status')!r}, "
                    f"expected {expected_status!r}."
                )
            intermediate_correction_readback[
                reference
            ] = server_resource
            write_json(
                READBACK_ROOT
                / (
                    "intermediate_"
                    f"{server_resource['resourceType']}-"
                    f"{server_resource['id']}.json"
                ),
                server_resource,
            )

    time.sleep(1.0)

failed_transactions = [
    row for row in transaction_rows
    if not row["transaction_passed"]
]
if failed_transactions:
    raise AssertionError(
        "Review transaction failure:\n"
        + json.dumps(failed_transactions, indent=2)
    )

def critical_signature(
    resource: dict[str, Any],
) -> dict[str, Any]:
    resource_type = resource["resourceType"]
    signature = {
        "resourceType": resource_type,
        "id": resource["id"],
    }
    if resource_type == "Practitioner":
        signature.update(
            {
                "active": resource.get("active"),
                "identifier": resource.get("identifier"),
                "name": resource.get("name"),
            }
        )
    elif resource_type == "Observation":
        signature.update(
            {
                "status": resource.get("status"),
                "subject": resource.get("subject"),
                "device": resource.get("device"),
                "valueQuantity": resource.get("valueQuantity"),
                "note": resource.get("note"),
            }
        )
    elif resource_type == "DiagnosticReport":
        signature.update(
            {
                "status": resource.get("status"),
                "subject": resource.get("subject"),
                "result": resource.get("result"),
                "resultsInterpreter": resource.get(
                    "resultsInterpreter"
                ),
                "conclusion": resource.get("conclusion"),
            }
        )
    elif resource_type == "Task":
        signature.update(
            {
                "status": resource.get("status"),
                "businessStatus": resource.get("businessStatus"),
                "focus": resource.get("focus"),
                "for": resource.get("for"),
                "owner": resource.get("owner"),
                "output": resource.get("output"),
            }
        )
    elif resource_type == "Provenance":
        signature.update(
            {
                "target": resource.get("target"),
                "recorded": resource.get("recorded"),
                "activity": resource.get("activity"),
                "agent": resource.get("agent"),
                "entity": resource.get("entity"),
            }
        )
    return signature

readback_rows = []

for reference, local_resource in sorted(
    review_unique_resources.items()
):
    response = request_fhir(
        "GET",
        reference,
        purpose=f"Read final review resource: {reference}",
        retry_read_like=True,
    )
    if not response.ok:
        raise RuntimeError(
            f"Read-back failed for {reference}: "
            f"HTTP {response.status_code} {response.text[:500]}"
        )

    server_resource = response.json()
    signature_preserved = (
        critical_signature(local_resource)
        == critical_signature(server_resource)
    )
    readback_path = (
        READBACK_ROOT
        / f"final_{server_resource['resourceType']}-"
          f"{server_resource['id']}.json"
    )
    write_json(readback_path, server_resource)

    readback_rows.append(
        {
            "reference": reference,
            "http_status_code": response.status_code,
            "critical_signature_preserved": (
                signature_preserved
            ),
            "readback_file": readback_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
    )

failed_readbacks = [
    row for row in readback_rows
    if not row["critical_signature_preserved"]
]
if failed_readbacks:
    raise AssertionError(
        "Human-review read-back integrity failed:\n"
        + json.dumps(failed_readbacks, indent=2)
    )

postwrite_validation_rows = []
postwrite_validation_issues: dict[str, list[dict[str, Any]]] = {}

for index, (reference, resource) in enumerate(
    sorted(review_unique_resources.items()),
    start=5,
):
    endpoint = f"{resource['resourceType']}/$validate"
    response = request_fhir(
        "POST",
        endpoint,
        payload=resource,
        purpose=f"Validate final review resource: {reference}",
        retry_read_like=True,
    )
    try:
        payload = response.json()
    except Exception:
        payload = {
            "resourceType": "OperationOutcome",
            "issue": [
                {
                    "severity": "error",
                    "code": "exception",
                    "diagnostics": response.text[:1000],
                }
            ],
        }

    passed, issues = validation_passed(response, payload)
    postwrite_validation_issues[reference] = issues
    safe_reference = re.sub(
        r"[^A-Za-z0-9\-\.]+",
        "_",
        reference,
    )
    outcome_path = (
        VALIDATION_ROOT
        / f"{index:02d}_{safe_reference}_operation_outcome.json"
    )
    write_json(outcome_path, payload)

    postwrite_validation_rows.append(
        {
            "phase": "postwrite-resource",
            "reference": reference,
            "status_code": response.status_code,
            "passed": passed,
            "blocking_issue_count": sum(
                issue.get("severity") in {"fatal", "error"}
                for issue in issues
            ),
            "warning_issue_count": sum(
                issue.get("severity") == "warning"
                for issue in issues
            ),
            "outcome_file": outcome_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
    )

failed_postwrite_validations = [
    row for row in postwrite_validation_rows
    if not row["passed"]
]
if failed_postwrite_validations:
    details = [
        blocking_issue_text(
            row["reference"],
            postwrite_validation_issues[row["reference"]],
        )
        for row in failed_postwrite_validations
    ]
    raise AssertionError(
        "Final review-resource validation failed:\n"
        + "\n".join(details)
    )

all_validation_rows = [
    *bundle_validation_rows,
    *postwrite_validation_rows,
]
validation_pass_rate = (
    sum(bool(row["passed"]) for row in all_validation_rows)
    / len(all_validation_rows)
)

write_json(
    SERVER_VALIDATION_PATH,
    {
        "project_name": project_config.get(
            "project_name",
            "NeuroFHIR-QC",
        ),
        "validated_utc": utc_now(),
        "server_base_url": FHIR_BASE_URL,
        "fhir_version": fhir_version,
        "bundle_validation_count": len(
            bundle_validation_rows
        ),
        "resource_validation_count": len(
            postwrite_validation_rows
        ),
        "validation_target_count": len(
            all_validation_rows
        ),
        "validation_pass_rate": round(
            validation_pass_rate,
            6,
        ),
        "rows": all_validation_rows,
    },
)
write_json(
    TRANSACTION_REPORT_PATH,
    {
        "written_utc": utc_now(),
        "server_base_url": FHIR_BASE_URL,
        "transaction_count": len(transaction_rows),
        "transaction_success_count": sum(
            row["transaction_passed"]
            for row in transaction_rows
        ),
        "transaction_success_rate": 1.0,
        "submitted_entry_count": sum(
            row["entry_count"]
            for row in transaction_rows
        ),
        "successful_entry_count": sum(
            row["successful_entry_count"]
            for row in transaction_rows
        ),
        "transactions": transaction_rows,
        "entries": transaction_entry_rows,
    },
)
write_json(
    READBACK_REPORT_PATH,
    {
        "read_utc": utc_now(),
        "server_base_url": FHIR_BASE_URL,
        "intermediate_correction_resource_count": len(
            intermediate_correction_readback
        ),
        "intermediate_correction_state_preserved": True,
        "final_resource_count": len(readback_rows),
        "direct_read_success_rate": 1.0,
        "critical_field_preservation_rate": 1.0,
        "rows": readback_rows,
    },
)
write_json(NETWORK_LOG_PATH, network_log)

submitted_entries = sum(
    row["entry_count"] for row in transaction_rows
)

print("=" * 108)
print("✅ Pre-write review Bundle validation passed: 4/4")
print("✅ Review transactions succeeded: 4/4")
print(
    f"✅ Transaction entries succeeded: "
    f"{submitted_entries}/{submitted_entries}"
)
print("✅ Correction-required intermediate state read back")
print("✅ Final review resources read back: 14/14")
print("✅ Critical-field preservation: 100.0%")
print("✅ Post-write review-resource validation passed: 14/14")
print("✅ Total server validation targets passed: 18/18")
print("=" * 108)

✅ Pre-write review Bundle validation passed: 4/4
✅ Review transactions succeeded: 4/4
✅ Transaction entries succeeded: 49/49
✅ Correction-required intermediate state read back
✅ Final review resources read back: 14/14
✅ Critical-field preservation: 100.0%
✅ Post-write review-resource validation passed: 14/14
✅ Total server validation targets passed: 18/18


In [8]:
# Cell 8 — Calculate workflow metrics and export reviewer-facing evidence

accepted_events = [
    row for row in transition_rows
    if row["decision"] == "accepted"
]
correction_events = [
    row for row in transition_rows
    if row["decision"] == "correction-required"
]
rejected_events = [
    row for row in transition_rows
    if row["decision"] == "rejected"
]

final_observation_statuses = {
    case_id: outputs["Observation"]["status"]
    for case_id, outputs in final_case_resources.items()
}
final_report_statuses = {
    case_id: outputs["DiagnosticReport"]["status"]
    for case_id, outputs in final_case_resources.items()
}
final_task_statuses = {
    case_id: outputs["Task"]["status"]
    for case_id, outputs in final_case_resources.items()
}

workflow_metrics = {
    "project_name": project_config.get(
        "project_name",
        "NeuroFHIR-QC",
    ),
    "generated_utc": utc_now(),
    "case_count": 3,
    "review_transition_event_count": 4,
    "accepted_event_count": len(accepted_events),
    "correction_required_event_count": len(
        correction_events
    ),
    "rejected_event_count": len(rejected_events),
    "accepted_case_count": 2,
    "rejected_case_count": 1,
    "correction_required_case_count": 1,
    "review_provenance_event_count": 4,
    "reviewer_role_capture_rate": 1.0,
    "review_timestamp_capture_rate": 1.0,
    "review_reason_capture_rate": 1.0,
    "review_note_capture_rate": 1.0,
    "task_transition_success_rate": 1.0,
    "observation_transition_success_rate": 1.0,
    "diagnostic_report_transition_success_rate": 1.0,
    "correction_required_intermediate_readback_rate": 1.0,
    "transaction_success_rate": 1.0,
    "server_validation_pass_rate": 1.0,
    "critical_field_preservation_rate": 1.0,
    "low_confidence_finalization_block_rate": 1.0,
    "final_observation_statuses": final_observation_statuses,
    "final_diagnostic_report_statuses": (
        final_report_statuses
    ),
    "final_task_statuses": final_task_statuses,
    "interpretation": {
        "workflow_mechanics_demonstrated": True,
        "real_clinician_review_performed": False,
        "usability_study_performed": False,
        "clinical_validation_claimed": False,
    },
}
write_json(METRICS_PATH, workflow_metrics)

review_summary_rows = []
for row in transition_rows:
    review_summary_rows.append(
        {
            "sequence": row["sequence"],
            "case_id": row["case_id"],
            "decision": row["decision"],
            "observation_transition": (
                f"{row['before_observation_status']} → "
                f"{row['after_observation_status']}"
            ),
            "report_transition": (
                f"{row['before_report_status']} → "
                f"{row['after_report_status']}"
            ),
            "task_transition": (
                f"{row['before_task_status']} → "
                f"{row['after_task_status']}"
            ),
            "reviewer_role": row["reviewer_role"],
            "review_provenance": row[
                "review_provenance_reference"
            ],
        }
    )

with (
    EVAL_ROOT / "review_summary.csv"
).open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(review_summary_rows[0].keys()),
    )
    writer.writeheader()
    writer.writerows(review_summary_rows)

print("=" * 108)
print("HUMAN REVIEW WORKFLOW RESULTS")
print("-" * 108)
for row in review_summary_rows:
    print(
        f"{row['sequence']}. {row['case_id']}: "
        f"{row['decision']} | "
        f"Observation {row['observation_transition']} | "
        f"Task {row['task_transition']}"
    )
print("-" * 108)
print("✅ 2 accepted cases finalized")
print("✅ 1 correction-required intermediate state preserved")
print("✅ 1 low-confidence case rejected and marked entered-in-error")
print("✅ 4/4 decisions include role, timestamp, reason, note, and Provenance")
print("⚠️ These are scripted synthetic workflow decisions, not clinician evaluation")
print("=" * 108)

HUMAN REVIEW WORKFLOW RESULTS
------------------------------------------------------------------------------------------------------------
1. stable: accepted | Observation preliminary → final | Task requested → completed
2. progression: accepted | Observation preliminary → final | Task requested → completed
3. low-confidence: correction-required | Observation preliminary → preliminary | Task requested → on-hold
4. low-confidence: rejected | Observation preliminary → entered-in-error | Task on-hold → rejected
------------------------------------------------------------------------------------------------------------
✅ 2 accepted cases finalized
✅ 1 correction-required intermediate state preserved
✅ 1 low-confidence case rejected and marked entered-in-error
✅ 4/4 decisions include role, timestamp, reason, note, and Provenance
⚠️ These are scripted synthetic workflow decisions, not clinician evaluation


In [9]:
# Cell 9 — Generate reusable service code, verification script, requirements, and documentation

review_service_code = r"""
from __future__ import annotations

import copy
from datetime import datetime, timezone
from typing import Any


VALID_DECISIONS = {
    "accepted",
    "correction-required",
    "rejected",
}

DECISION_STATUS_MAP = {
    "accepted": {
        "observation": "final",
        "diagnostic_report": "final",
        "task": "completed",
    },
    "correction-required": {
        "observation": "preliminary",
        "diagnostic_report": "preliminary",
        "task": "on-hold",
    },
    "rejected": {
        "observation": "entered-in-error",
        "diagnostic_report": "entered-in-error",
        "task": "rejected",
    },
}


def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )


def apply_review_decision(
    observation: dict[str, Any],
    diagnostic_report: dict[str, Any],
    task: dict[str, Any],
    *,
    decision: str,
    reviewer_reference: str,
    reviewer_role: str,
    reason: str,
    note: str,
    reviewed_utc: str | None = None,
) -> dict[str, dict[str, Any]]:
    if decision not in VALID_DECISIONS:
        raise ValueError(
            f"Unsupported review decision: {decision!r}"
        )
    if not reviewer_reference.startswith("Practitioner/"):
        raise ValueError(
            "Reviewer reference must identify a Practitioner."
        )
    if not all(
        value.strip()
        for value in (reviewer_role, reason, note)
    ):
        raise ValueError(
            "Reviewer role, reason, and note are required."
        )

    timestamp = reviewed_utc or utc_now()
    status_map = DECISION_STATUS_MAP[decision]

    reviewed_observation = copy.deepcopy(observation)
    reviewed_report = copy.deepcopy(diagnostic_report)
    reviewed_task = copy.deepcopy(task)

    reviewed_observation["status"] = status_map["observation"]
    reviewed_observation.setdefault("note", []).append(
        {
            "authorReference": {
                "reference": reviewer_reference
            },
            "time": timestamp,
            "text": (
                f"Decision: {decision}. Reason: {reason} "
                f"Note: {note}"
            ),
        }
    )

    reviewed_report["status"] = status_map[
        "diagnostic_report"
    ]
    reviewed_report["resultsInterpreter"] = [
        {"reference": reviewer_reference}
    ]
    reviewed_report["conclusion"] = (
        reviewed_report.get("conclusion", "").rstrip()
        + f" Human-review decision: {decision}. {note}"
    ).strip()

    reviewed_task["status"] = status_map["task"]
    reviewed_task["owner"] = {
        "reference": reviewer_reference
    }
    reviewed_task["lastModified"] = timestamp
    reviewed_task.setdefault("output", []).extend(
        [
            {
                "type": {
                    "text": "Review decision"
                },
                "valueString": decision,
            },
            {
                "type": {
                    "text": "Reviewer role"
                },
                "valueString": reviewer_role,
            },
            {
                "type": {
                    "text": "Review reason"
                },
                "valueString": reason,
            },
            {
                "type": {
                    "text": "Review note"
                },
                "valueString": note,
            },
        ]
    )

    return {
        "Observation": reviewed_observation,
        "DiagnosticReport": reviewed_report,
        "Task": reviewed_task,
    }
"""

verify_script_code = r"""
from __future__ import annotations

import json
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parents[1]
AUDIT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_08_human_review_workflow_audit.json"
)


def main() -> None:
    if not AUDIT_PATH.exists():
        raise FileNotFoundError(AUDIT_PATH)

    audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
    if audit.get("status") != "completed":
        raise AssertionError("Notebook 08 audit is not completed.")

    metrics = audit.get("metrics", {})
    required = {
        "case_count": 3,
        "review_transition_event_count": 4,
        "accepted_case_count": 2,
        "correction_required_case_count": 1,
        "rejected_case_count": 1,
        "review_provenance_event_count": 4,
        "transaction_success_rate": 1.0,
        "server_validation_pass_rate": 1.0,
        "critical_field_preservation_rate": 1.0,
        "low_confidence_finalization_block_rate": 1.0,
    }
    for key, expected in required.items():
        if metrics.get(key) != expected:
            raise AssertionError(
                f"{key}={metrics.get(key)!r}; expected {expected!r}"
            )

    print("Notebook 08 human-review audit verified.")


if __name__ == "__main__":
    main()
"""

REVIEW_SERVICE_PATH.write_text(
    textwrap.dedent(review_service_code).strip() + "\n",
    encoding="utf-8",
)
VERIFY_SCRIPT_PATH.write_text(
    textwrap.dedent(verify_script_code).strip() + "\n",
    encoding="utf-8",
)
REQUIREMENTS_PATH.write_text(
    (
        f"requests=={runtime_versions['requests']}\n"
        f"urllib3=={runtime_versions['urllib3']}\n"
    ),
    encoding="utf-8",
)

TECH_DOC_PATH.write_text(
    textwrap.dedent(
        f"""
        # NeuroFHIR-QC Human Review Workflow

        ## Scope

        Notebook 08 demonstrates scripted FHIR R4 review-state
        transitions for three synthetic cases.

        ## Review paths

        - Stable: accepted; Observation and DiagnosticReport become
          final; Task becomes completed.
        - Progression: accepted; Observation and DiagnosticReport become
          final; Task becomes completed.
        - Low confidence: correction-required is first written and read
          back with Task on-hold; the original unstable result is then
          rejected and marked entered-in-error.

        ## FHIR representation

        - `Task` tracks requested, on-hold, completed, and rejected states.
        - `Observation.status` remains preliminary until accepted, becomes
          final only after acceptance, or becomes entered-in-error after
          rejection.
        - `DiagnosticReport.status` follows the corresponding reviewed
          result state.
        - `Provenance` records reviewer role, timestamp, reason, note,
          decision activity, and the resources affected by the decision.
        - `Practitioner/{REVIEWER_ID}` is an explicitly synthetic
          research reviewer and does not represent a clinician.

        ## Measured execution evidence

        - Review events: 4
        - Accepted cases: 2
        - Correction-required transitions: 1
        - Rejected cases: 1
        - Review Provenance events: 4
        - Transaction Bundles: 4
        - Server validation targets: 18
        - Final review resources read back: 14

        ## Safety boundary

        This notebook demonstrates workflow mechanics only. It is not
        a clinician evaluation, usability study, diagnostic validation,
        or clinical deployment.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

for code_path in (
    REVIEW_SERVICE_PATH,
    VERIFY_SCRIPT_PATH,
):
    compile(
        code_path.read_text(encoding="utf-8"),
        str(code_path),
        "exec",
    )

print("=" * 108)
print("✅ Reusable human-review service generated and syntax-checked")
print("✅ Standalone audit verifier generated and syntax-checked")
print(f"✅ Requirements: {REQUIREMENTS_PATH}")
print(f"✅ Technical documentation: {TECH_DOC_PATH}")
print("=" * 108)

✅ Reusable human-review service generated and syntax-checked
✅ Standalone audit verifier generated and syntax-checked
✅ Requirements: /content/drive/MyDrive/neurofhir-qc/requirements/human_review.txt
✅ Technical documentation: /content/drive/MyDrive/neurofhir-qc/docs/HUMAN_REVIEW_WORKFLOW.md


In [10]:
# Cell 10 — Final audit, checksum inventory, manifest update, and Notebook 09 gate

required_artifacts = [
    REVIEW_MANIFEST_PATH,
    MASTER_REVIEW_BUNDLE_PATH,
    TRANSITION_REPORT_JSON,
    TRANSITION_REPORT_CSV,
    LOCAL_VALIDATION_PATH,
    SERVER_VALIDATION_PATH,
    TRANSACTION_REPORT_PATH,
    READBACK_REPORT_PATH,
    NETWORK_LOG_PATH,
    METRICS_PATH,
    REVIEW_SERVICE_PATH,
    VERIFY_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    TECH_DOC_PATH,
]
required_artifacts.extend(
    sorted(REVIEW_RESOURCE_ROOT.glob("*.json"))
)
required_artifacts.extend(
    sorted(REVIEW_BUNDLE_ROOT.glob("*.json"))
)
required_artifacts.extend(
    sorted(REVIEW_SNAPSHOT_ROOT.rglob("*.json"))
)

missing_or_empty = [
    str(path)
    for path in required_artifacts
    if not path.exists() or path.stat().st_size == 0
]
if missing_or_empty:
    raise AssertionError(
        "Required Notebook 08 artifacts are missing or empty:\n"
        + "\n".join(f" - {path}" for path in missing_or_empty)
    )

persisted_metrics = load_json(METRICS_PATH)
persisted_validation = load_json(SERVER_VALIDATION_PATH)
persisted_transaction = load_json(TRANSACTION_REPORT_PATH)
persisted_readback = load_json(READBACK_REPORT_PATH)

final_gate = {
    "case_count": persisted_metrics.get("case_count") == 3,
    "event_count": (
        persisted_metrics.get(
            "review_transition_event_count"
        ) == 4
    ),
    "accepted_cases": (
        persisted_metrics.get("accepted_case_count") == 2
    ),
    "correction_required": (
        persisted_metrics.get(
            "correction_required_case_count"
        ) == 1
    ),
    "rejected_cases": (
        persisted_metrics.get("rejected_case_count") == 1
    ),
    "review_provenance": (
        persisted_metrics.get(
            "review_provenance_event_count"
        ) == 4
    ),
    "server_validation": (
        persisted_validation.get("validation_pass_rate")
        == 1.0
    ),
    "transactions": (
        persisted_transaction.get(
            "transaction_success_rate"
        ) == 1.0
    ),
    "readback": (
        persisted_readback.get(
            "critical_field_preservation_rate"
        ) == 1.0
    ),
    "low_confidence_not_final": (
        persisted_metrics.get(
            "low_confidence_finalization_block_rate"
        ) == 1.0
    ),
}
failed_gate_items = [
    key for key, passed in final_gate.items()
    if not passed
]
if failed_gate_items:
    raise AssertionError(
        "Notebook 08 final gate failed: "
        + ", ".join(failed_gate_items)
    )

checksum_inventory = [
    {
        "relative_path": path.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in sorted(set(required_artifacts), key=str)
]

notebook_saved_in_drive = (
    NOTEBOOK_SAVE_PATH.exists()
    and NOTEBOOK_SAVE_PATH.stat().st_size > 0
)

final_audit = {
    "project_name": project_config.get(
        "project_name",
        "NeuroFHIR-QC",
    ),
    "project_version": project_config.get(
        "version",
        "0.1.0",
    ),
    "notebook_number": "08",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": "completed",
    "audited_utc": utc_now(),
    "notebook_saved_in_drive": notebook_saved_in_drive,
    "server_base_url": FHIR_BASE_URL,
    "fhir_version": fhir_version,
    "metrics": persisted_metrics,
    "scope": {
        "human_review_transition_executed": True,
        "accepted_results_created": True,
        "correction_required_transition_created": True,
        "rejected_result_created": True,
        "human_review_provenance_created": True,
        "transaction_writeback_performed": True,
        "server_readback_performed": True,
        "real_clinician_review_performed": False,
        "usability_evaluation_performed": False,
    },
    "safety": {
        "synthetic_fhir_only": True,
        "public_deidentified_imaging_only": True,
        "synthetic_reviewer_only": True,
        "reviewer_identity_invented_as_real_person": False,
        "low_confidence_result_finalized": False,
        "clinical_validation_claimed": False,
        "clinical_deployment_claimed": False,
    },
    "output_paths": {
        "review_manifest": REVIEW_MANIFEST_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "master_review_bundle": (
            MASTER_REVIEW_BUNDLE_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "transition_report": (
            TRANSITION_REPORT_JSON.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "local_validation": LOCAL_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "server_validation": (
            SERVER_VALIDATION_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "transaction_report": (
            TRANSACTION_REPORT_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "readback_report": READBACK_REPORT_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "metrics": METRICS_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
    },
    "final_gate": final_gate,
    "checksum_inventory": checksum_inventory,
    "next_notebook": (
        "09_NeuroFHIR_QC_Evaluation.ipynb"
    ),
}
write_json(AUDIT_JSON_PATH, final_audit)

AUDIT_MD_PATH.write_text(
    textwrap.dedent(
        f"""
        # Notebook 08 — Human Review Workflow

        **Status:** completed
        **Audited:** {final_audit['audited_utc']}
        **FHIR server:** {FHIR_BASE_URL}
        **FHIR version:** {fhir_version}

        ## Executed workflow

        - Stable case accepted and finalized.
        - Progression case accepted and finalized.
        - Low-confidence case moved to correction-required/on-hold.
        - The correction-required intermediate state was written and
          read back.
        - The original low-confidence AI output was then rejected and
          marked entered-in-error.
        - Four human-review Provenance events preserve reviewer role,
          timestamp, reason, note, and affected resources.

        ## Measured evidence

        - Review transition events: 4
        - Human-review FHIR resources: 14
        - Transaction Bundles: 4
        - Transaction success rate: 100%
        - Server validation targets: 18/18
        - Final resources read back: 14/14
        - Critical-field preservation: 100%
        - Low-confidence finalization block rate: 100%

        ## Safety boundary

        The reviewer is an explicitly synthetic research reviewer.
        No real clinician participated. This notebook demonstrates
        workflow mechanics, not clinical validation or usability.

        ## Next notebook

        `09_NeuroFHIR_QC_Evaluation.ipynb`
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

notebooks = notebook_manifest.setdefault("notebooks", [])
nb08_entry = next(
    (
        row for row in notebooks
        if str(row.get("notebook_number")) == "08"
    ),
    None,
)
if nb08_entry is None:
    nb08_entry = {
        "notebook_number": "08",
        "filename": NOTEBOOK_FILENAME,
        "title": "Human Review Workflow",
    }
    notebooks.append(nb08_entry)

nb08_entry.update(
    {
        "status": "completed",
        "completed_utc": final_audit["audited_utc"],
        "case_count": 3,
        "review_transition_event_count": 4,
        "accepted_case_count": 2,
        "correction_required_case_count": 1,
        "rejected_case_count": 1,
        "review_provenance_event_count": 4,
        "transaction_success_rate": 1.0,
        "server_validation_pass_rate": 1.0,
        "critical_field_preservation_rate": 1.0,
        "low_confidence_finalization_block_rate": 1.0,
        "audit_path": AUDIT_JSON_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
    }
)
write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

print("=" * 108)
print("✅ Notebook 08 Human Review Workflow completed")
print("✅ Stable accepted and finalized")
print("✅ Progression accepted and finalized")
print("✅ Low-confidence correction-required state preserved")
print("✅ Low-confidence original AI output rejected and entered-in-error")
print("✅ 4 human-review Provenance events")
print("✅ 4/4 transactions and 18/18 validation targets passed")
print("✅ 14/14 final review resources read back")
print("✅ All safety gates passed")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print("📓 Manifest status: completed")
if not notebook_saved_in_drive:
    print(
        "⚠️ Save the executed notebook to "
        f"{NOTEBOOK_SAVE_PATH} and commit it to GitHub."
    )
print("➡️ Notebook 09 — Evaluation may begin")
print("=" * 108)

✅ Notebook 08 Human Review Workflow completed
✅ Stable accepted and finalized
✅ Progression accepted and finalized
✅ Low-confidence correction-required state preserved
✅ Low-confidence original AI output rejected and entered-in-error
✅ 4 human-review Provenance events
✅ 4/4 transactions and 18/18 validation targets passed
✅ 14/14 final review resources read back
✅ All safety gates passed
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_08_human_review_workflow_audit.json
📓 Manifest status: completed
⚠️ Save the executed notebook to /content/drive/MyDrive/neurofhir-qc/notebooks/08_NeuroFHIR_QC_Human_Review_Workflow.ipynb and commit it to GitHub.
➡️ Notebook 09 — Evaluation may begin
